# NUMPY MASTERY — CHAPTERS 66–75 (COURSE FINALE)

```
NUMPY MASTERY
Current Chapters: 66 (Einsum), 67 (Tensor Ops), 68 (Advanced Simulation),
                  69 (Scientific Computing Ecosystem), 70 (Architecture/Internals),
                  71 (System Design Scenarios), 72 (Interview Traps),
                  73 (Cheat Sheet), 74 (Capstone), 75 (Mock Interview)
Progress:         75 / 75 chapters — COURSE COMPLETE
Completed:        Ch1-65
Remaining:        —
```

The last ten chapters, together. 66-68 are hands-on code; 69-73 are reference/judgment
material meant to be reread before interviews, not just read once; 74 is a real assignment —
attempt it yourself, don't just read the spec; 75 opens the mock interview format for you to
actually use in a follow-up conversation, since a mock interview only works as a real dialogue.


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


---
# CHAPTER 66 — Einsum Mastery

## Concept
`np.einsum` expresses sums-of-products over indexed axes using Einstein summation notation — a
single, unified syntax for dot products, matrix multiplication, transposes, traces, batch
operations, and tensor contractions that would otherwise need many different function calls.

## How to read einsum notation
`np.einsum('ij,jk->ik', A, B)` means: "for output index `(i,k)`, sum over `j` the product of
`A[i,j]` and `B[j,k]`" — an index that appears in the inputs but NOT in the output (here, `j`)
is **summed over** (contracted); an index that appears in the output is **kept**.


In [ ]:
A = np.array([[1., 2.], [3., 4.]])
B = np.array([[5., 6.], [7., 8.]])

# --- Matrix multiplication ---
matmul_einsum = np.einsum('ij,jk->ik', A, B)
matmul_at = A @ B
print("einsum matmul == A @ B?", np.allclose(matmul_einsum, matmul_at))

# --- Transpose ---
transpose_einsum = np.einsum('ij->ji', A)
print("einsum transpose == A.T?", np.array_equal(transpose_einsum, A.T))

# --- Trace (sum of diagonal) ---
trace_einsum = np.einsum('ii->', A)          # repeated index, NOT in output -> sum the diagonal
print("einsum trace == np.trace(A)?", trace_einsum == np.trace(A))

# --- Dot product ---
u, v = np.array([1., 2., 3.]), np.array([4., 5., 6.])
dot_einsum = np.einsum('i,i->', u, v)
print("einsum dot == np.dot(u,v)?", dot_einsum == np.dot(u, v))

# --- Outer product ---
outer_einsum = np.einsum('i,j->ij', u, v)
print("einsum outer == np.outer(u,v)?", np.array_equal(outer_einsum, np.outer(u, v)))

# --- Element-wise multiplication then sum along one axis (a common ML pattern) ---
elementwise_then_sum = np.einsum('ij,ij->i', A, B)   # sum(A*B, axis=1), per row
print("\neinsum row-wise (A*B).sum(1):", elementwise_then_sum, " matches:", np.array_equal(elementwise_then_sum, (A * B).sum(axis=1)))


### Expected Output (verified on NumPy 2.4.4)
```
einsum matmul == A @ B? True
einsum transpose == A.T? True
einsum trace == np.trace(A)? True
einsum dot == np.dot(u,v)? True
einsum outer == np.outer(u,v)? True

einsum row-wise (A*B).sum(1): [17. 53.] matches: True
```

### Why `'ii->'` computes a trace
`ii` uses the **same** letter for both axes of a 2D input — this forces `i` to walk the
diagonal only (where row index equals column index); since `i` doesn't appear in the output
(`->` followed by nothing), it's summed over, giving exactly `sum(A[i,i] for all i)` — the
trace, expressed without ever calling `np.trace` or `np.diag`.

### Batched matrix multiplication — where einsum earns its keep in deep learning code


In [ ]:
batch_A = np.random.default_rng(0).normal(size=(5, 2, 3))   # 5 batches of (2,3)
batch_B = np.random.default_rng(1).normal(size=(5, 3, 4))   # 5 batches of (3,4)

batch_matmul_einsum = np.einsum('bij,bjk->bik', batch_A, batch_B)   # 'b' kept as batch axis
batch_matmul_at = batch_A @ batch_B                                    # Chapter 22's batched @

print("einsum batched matmul shape:", batch_matmul_einsum.shape)
print("Matches batch_A @ batch_B? ", np.allclose(batch_matmul_einsum, batch_matmul_at))

# --- Attention-style score computation: THE canonical einsum use case in modern deep learning ---
# queries: (batch, seq_len, d), keys: (batch, seq_len, d) -> scores: (batch, seq_len, seq_len)
queries = np.random.default_rng(2).normal(size=(2, 4, 8))
keys = np.random.default_rng(3).normal(size=(2, 4, 8))
attention_scores = np.einsum('bqd,bkd->bqk', queries, keys)
print("\nAttention scores shape:", attention_scores.shape, " <- (batch, query_pos, key_pos)")
print("Matches manual @ with transpose?",
      np.allclose(attention_scores, queries @ keys.transpose(0, 2, 1)))


### Expected Output (verified on NumPy 2.4.4)
```
einsum batched matmul shape: (5, 2, 4)
Matches batch_A @ batch_B?  True

Attention scores shape: (2, 4, 4)  <- (batch, query_pos, key_pos)
Matches manual @ with transpose? True
```

### Why einsum genuinely wins for the attention case
The `@`/`matmul` equivalent (`queries @ keys.transpose(0, 2, 1)`) requires *manually* figuring
out which axes to transpose before the shapes align for batched matmul — easy to get subtly
wrong for complex tensor contractions. `np.einsum('bqd,bkd->bqk', queries, keys)` states the
*intended contraction* directly (sum over the shared `d` dimension, keep `b`, `q`, `k`) without
requiring any manual transpose reasoning — this is exactly why transformer/attention
implementations across PyTorch, TensorFlow, and JAX rely on einsum-style notation heavily: it
scales to arbitrarily complex tensor contractions without the transpose-juggling that `@` alone
would require.

### `einsum` vs `matmul` vs `dot` — readability/performance tradeoffs
| | Readability for complex contractions | Performance | Best for |
|---|---|---|---|
| `@`/`matmul` | Poor beyond simple batched 2D cases (needs manual transposes) | Usually fastest (highly optimized BLAS path) | Standard 2D/batched matrix multiplication |
| `np.dot` | Poor for anything beyond 1D/2D | Comparable to matmul for those cases | 1D dot products, simple 2D matmul |
| `einsum` | Best — states the contraction directly, any complexity | Can be slower than `matmul` for simple cases (though NumPy optimizes common einsum patterns internally) | Complex/unusual tensor contractions, batch operations with nonstandard axis arrangements |

### Interview Insight
> "When would you reach for `einsum` instead of `@`/`matmul`?"
When the desired contraction doesn't map cleanly onto standard batched matrix multiplication
without manual transposing — einsum lets you specify exactly which axes are summed and which
are kept, directly, which is both less error-prone and more readable for complex operations
like attention score computation, especially as the number of batch/head dimensions grows.


---
# CHAPTER 67 — Tensor Operations

## Concept
`np.tensordot` generalizes matrix multiplication to contract **arbitrary** axes of two
arrays, not just "last axis of A with second-to-last of B" — useful when a contraction doesn't
naturally fit the standard matmul shape.


In [ ]:
A = np.arange(24).reshape(2, 3, 4)
B = np.arange(20).reshape(4, 5)

# Contract A's LAST axis (size 4) with B's FIRST axis (size 4) -- this is a matmul-LIKE contraction
result = np.tensordot(A, B, axes=([2], [0]))
print("A.shape:", A.shape, " B.shape:", B.shape)
print("tensordot(A, B, axes=([2],[0])).shape:", result.shape, " <- (2,3,5): the contracted axes VANISH")

# Compare against einsum expressing the SAME contraction
result_einsum = np.einsum('ijk,kl->ijl', A, B)
print("Matches einsum equivalent?", np.array_equal(result, result_einsum))

# Contracting TWO axes at once (a genuinely tensordot-specific convenience)
C = np.random.default_rng(0).normal(size=(3, 4, 5))
D = np.random.default_rng(1).normal(size=(4, 5, 6))
result2 = np.tensordot(C, D, axes=([1, 2], [0, 1]))   # contract BOTH matching axis pairs at once
print("\nC.shape:", C.shape, " D.shape:", D.shape)
print("tensordot contracting 2 axis pairs -> shape:", result2.shape, " <- (3,6)")


### Expected Output (verified on NumPy 2.4.4)
```
A.shape: (2, 3, 4)  B.shape: (4, 5)
tensordot(A, B, axes=([2],[0])).shape: (2, 3, 5)  <- (2,3,5): the contracted axes VANISH
Matches einsum equivalent? True

C.shape: (3, 4, 5)  D.shape: (4, 5, 6)
tensordot contracting 2 axis pairs -> shape: (3, 6)
```

### Reading `axes=([1,2],[0,1])` for the double-contraction case
This tells `tensordot` to sum over **both** `C`'s axis 1 (size 4) matched with `D`'s axis 0
(size 4), **and** `C`'s axis 2 (size 5) matched with `D`'s axis 1 (size 5) — simultaneously.
The remaining un-contracted axes (`C`'s axis 0, size 3; `D`'s axis 2, size 6) form the output
shape `(3, 6)`. This specific "contract two axis pairs at once" pattern would require a more
verbose `einsum` string (`'ijk,jkl->il'`, equally valid but a different notation for the exact
same operation) — `tensordot` and `einsum` are two notations for overlapping capability, and
knowing both means picking whichever reads more clearly for a given contraction.

### Real-world use
Computing a batch of convolution-like operations, contracting a weight tensor against multiple
input dimensions simultaneously (common in tensor-factorization models), or reducing a
higher-order tensor along specific axes without first reshaping/flattening it — `tensordot`
avoids intermediate reshape calls when the contraction pattern fits its `axes=` specification
directly.


---
# CHAPTER 68 — Advanced Numerical Simulation

## Concept
Beyond Chapter 21's Monte Carlo introduction: vectorized random walks and portfolio simulation
— generating and analyzing thousands of simulated paths simultaneously, a single 2D array
representing (path, timestep) rather than looping over paths.


In [ ]:
rng = np.random.default_rng(0)

# --- Vectorized random walk: ALL paths, ALL timesteps, in one array ---
n_paths = 10_000
n_steps = 252   # e.g., trading days in a year

steps = rng.choice([-1, 1], size=(n_paths, n_steps))    # (n_paths, n_steps) -- every step, every path
paths = np.cumsum(steps, axis=1)                          # Chapter 10's axis: cumsum ALONG each path's timeline
paths = np.column_stack([np.zeros(n_paths), paths])         # prepend starting position 0

print("paths.shape:", paths.shape, " <- (n_paths, n_steps+1), fully vectorized, zero Python loops")
print("Final position -- mean:", paths[:, -1].mean(), " std:", round(paths[:, -1].std(), 2))
print("(Theory: mean should be ~0, std should be ~sqrt(n_steps) =", round(np.sqrt(n_steps), 2), ")")

# --- Portfolio simulation: vectorized geometric Brownian motion for asset price paths ---
n_sims = 5000
n_days = 252
mu, sigma, S0 = 0.08, 0.20, 100.0   # annual drift, annual volatility, starting price
dt = 1 / 252

daily_returns = rng.normal((mu - 0.5 * sigma**2) * dt, sigma * np.sqrt(dt), size=(n_sims, n_days))
log_price_paths = np.log(S0) + np.cumsum(daily_returns, axis=1)
price_paths = np.exp(log_price_paths)

final_prices = price_paths[:, -1]
print("\nSimulated final prices after 1 year -- mean:", round(final_prices.mean(), 2),
      " median:", round(np.median(final_prices), 2))
var_95 = np.percentile(final_prices, 5)   # Chapter 14's percentile, applied to risk (Value at Risk)
print(f"5th percentile (95% VaR threshold): {var_95:.2f}  <- 95% of simulated outcomes exceed this price")


### Expected Output (verified on NumPy 2.4.4)
```
paths.shape: (10000, 253)  <- (n_paths, n_steps+1), fully vectorized, zero Python loops
Final position -- mean: 0.1336  std: 15.69
(Theory: mean should be ~0, std should be ~sqrt(n_steps) = 15.87 )

Simulated final prices after 1 year -- mean: 108.29  median: 106.48
5th percentile (95% VaR threshold): 76.66
```

### Reading the random walk validation
The simulated mean (`~0.13`, close to the theoretical `0`) and std (`~15.69`, very close to the
theoretical `√252 ≈ 15.87`) matching known random-walk statistics is a genuine correctness
check — this is the same "verify against known theory" discipline used throughout the course
(Chapter 38's `lstsq` cross-check, Chapter 46's gradient check), now applied to a simulation.

### The Value-at-Risk (VaR) calculation, read carefully
`np.percentile(final_prices, 5)` finds the price below which only 5% of simulated outcomes
fall — a standard risk metric: "there's a 95% chance the asset price after one year exceeds
this value, and a 5% chance it's worse." This single line, built entirely from Chapter 14's
percentile function applied to Chapter 20/21's vectorized random sampling, is a real,
production-relevant financial risk calculation, not just a toy example.

### Why this is 10,000x-fewer-lines than a looped simulation
Simulating `10,000` random walk paths with a Python loop would mean `10,000` separate calls
generating `252` steps each, accumulating manually — here, `rng.choice(..., size=(n_paths,
n_steps))` generates the **entire** random matrix in one call, and `np.cumsum(..., axis=1)`
computes all path trajectories simultaneously — this is Chapter 1's foundational lesson,
scaled up to a genuinely industrial-strength simulation.


---
# CHAPTER 69 — NumPy's Place in the Scientific Computing Ecosystem

## Concept
NumPy is the shared numerical foundation nearly every other Python scientific/ML library is
built on. Knowing which library owns which layer prevents both "reinventing the wheel in raw
NumPy" and "reaching for a heavy dependency when NumPy alone would do."

| Library | Built on NumPy? | Owns | Reach for it when |
|---|---|---|---|
| **SciPy** | Yes | Advanced algorithms: optimization, signal processing, sparse matrices, statistics, ODE solvers | You need `scipy.optimize.minimize`, `scipy.stats` distributions beyond `np.random`, or sparse matrix support |
| **scikit-learn** | Yes | Production-grade, tested implementations of classical ML (the algorithms Chapters 38-42 built from scratch) | Production model training — never re-derive linear/logistic regression, KNN, PCA from scratch for real use; this course built them so you understand what scikit-learn does internally |
| **statsmodels** | Yes | Statistical modeling with a strong emphasis on inference (p-values, confidence intervals, hypothesis tests) | You need statistical significance testing, not just point predictions |
| **OpenCV** | Yes (images as NumPy arrays) | Optimized, hardware-accelerated image/video processing, computer vision algorithms | Production image pipelines beyond Chapter 43's basic array operations — edge detection, feature matching, video I/O |
| **PyTorch / TensorFlow** | Conceptually parallel (Chapter 55) | GPU computation, automatic differentiation, deep learning training infrastructure | Training neural networks at real scale — Chapter 46 built the mechanism by hand so `.backward()` isn't a black box |
| **JAX** | Conceptually parallel, NumPy-compatible API (`jax.numpy`) | Automatic differentiation + JIT compilation + GPU/TPU, with an API deliberately mirroring NumPy | Research code needing autograd with NumPy-like syntax, or JIT-compiled numerical code |

## The general principle
NumPy is rarely the *final* tool for a production task — it's the substrate everything else is
built on and interfaces through. Recognizing "this is a `scipy.optimize` problem" or "this is a
`scikit-learn` problem" rather than re-deriving a solved algorithm in raw NumPy is itself a
sign of engineering maturity — this course built things from scratch specifically so you'd
understand what these libraries do internally, not so you'd avoid using them in practice.

### Interview Insight
> "You've built linear regression, PCA, and a neural network from scratch in NumPy. Would you
> use these implementations in a production system?"
No — production code should use scikit-learn, PyTorch, or equivalent battle-tested libraries,
which handle numerical edge cases, performance optimization, and correctness far more robustly
than a from-scratch implementation built for learning purposes. The value of building them from
scratch is understanding exactly what's happening inside those libraries — being able to debug
unexpected behavior, reason about numerical stability, and make informed architecture decisions
— not replacing the libraries themselves.


---
# CHAPTER 70 — NumPy Architecture / Internals

## Concept
A consolidated mental model of what happens between writing `arr = np.array([1,2,3])` and the
bytes existing in memory — assembling pieces from Chapters 1, 3, 12, and 28 into one picture.

```
Python object (np.array call)
        |
        v
dtype inference/resolution        <- Chapter 4: what type should every element be?
        |
        v
Memory allocation (ONE contiguous block, sized itemsize * n_elements)   <- Chapter 3
        |
        v
ndarray object created: a lightweight HEADER containing
    - pointer to the data buffer
    - shape tuple
    - strides tuple                <- Chapter 28: bytes-to-step-per-axis
    - dtype
    - flags (C_CONTIGUOUS, OWNDATA, WRITEABLE, ...)
        |
        v
Operations (arr + 1, arr @ other, arr.sum()) dispatch to UFUNCS  <- Chapter 12
    - compiled C loops, chosen based on dtype
    - broadcasting rules resolved BEFORE the loop runs            <- Chapter 11
    - loop walks memory using strides, NOT Python-level indexing
        |
        v
Result: either a VIEW (new header, same buffer) or a COPY (new header, new buffer)  <- Chapter 8
```

## Why this matters practically, not just academically
Every performance and memory lesson in this course (Chapters 28-31) is a direct consequence of
this architecture: contiguous memory enables cache-friendly compiled loops; strides make
transpose/reshape "free" when possible; the view/copy distinction exists because sometimes a
new header pointing at old data is sufficient, and sometimes new data must genuinely be
allocated.


In [ ]:
arr = np.array([1, 2, 3, 4, 5, 6], dtype=np.int32).reshape(2, 3)

print("The ndarray HEADER, made visible:")
print(f"  data buffer pointer (via .__array_interface__): {arr.__array_interface__['data'][0]}")
print(f"  shape   : {arr.shape}")
print(f"  strides : {arr.strides}")
print(f"  dtype   : {arr.dtype}")
print(f"  itemsize: {arr.itemsize} bytes")
print(f"  flags C_CONTIGUOUS: {arr.flags['C_CONTIGUOUS']}")
print(f"  flags OWNDATA     : {arr.flags['OWNDATA']}")

view = arr.T   # a transpose -- NEW header, SAME buffer
print(f"\nAfter transpose (arr.T):")
print(f"  view's data pointer: {view.__array_interface__['data'][0]}  <- SAME address as arr's, confirmed below")
print(f"  Same buffer as original?", view.__array_interface__['data'][0] == arr.__array_interface__['data'][0])
print(f"  view's shape/strides changed instead: shape={view.shape}, strides={view.strides}")


### Expected Output (exact pointer address varies by run — the EQUALITY is what matters)
```
The ndarray HEADER, made visible:
  data buffer pointer (via .__array_interface__): 628598880
  shape   : (2, 3)
  strides : (12, 4)
  dtype   : int32
  itemsize: 4 bytes
  flags C_CONTIGUOUS: True
  flags OWNDATA     : False

After transpose (arr.T):
  view's data pointer: 628598880  <- SAME address as arr's, confirmed below
  Same buffer as original?  True
  view's shape/strides changed instead: shape=(3, 2), strides=(4, 12)
```

### A bonus, unplanned demonstration: `OWNDATA` is `False` even for `arr` itself
Notice `arr.flags['OWNDATA']` prints `False`, not `True` — because `arr` was built as
`np.array([...]).reshape(2, 3)`, and `.reshape()` (Chapter 9) returned a **view** of the
intermediate 1D array `np.array([...])` rather than a fresh allocation. `arr` is a header
pointing at memory it doesn't itself own; the original 1D array (now out of scope, but kept
alive by `arr`'s reference to its buffer) is the true owner. This is a real, easy-to-miss
consequence of Chapter 8's view mechanism: a chain of "free" view-producing operations
(`reshape`, `.T`, slicing) can leave you several links away from whichever object actually owns
the underlying memory — `.base` (not shown here, but covered in Chapter 8) lets you walk that
chain to find it.

### This is Chapter 8 and Chapter 28, made literally visible
The transpose's data pointer is **identical** to the original's — not "conceptually the same
data," but the literal same memory address. Only the **header** (shape and strides) changed —
from `(2,3)`/`(12,4)` to `(3,2)`/`(4,12)`, exactly swapped. This is the concrete, addressable
proof behind every "transpose is free, it's just metadata" claim made throughout this course.

### Interview Insight
> "Explain, at a mechanical level, why `arr.T` doesn't copy data."
`ndarray` is a lightweight header (pointer to a data buffer, shape, strides, dtype, flags)
separate from the actual data buffer it points to. Transposing only needs to swap the shape and
strides tuples in a *new* header pointing at the *same* underlying buffer — reading the
identical bytes in a different logical order — so no data movement is required, only metadata
construction, which is why it's effectively instantaneous regardless of array size.


---
# CHAPTER 71 — NumPy System Design Scenarios

Attempt each scenario yourself before reading the model answer — these are deliberately
open-ended, the way a real system-design interview question would be.

## Scenario 1: 500 million numerical records
**Question:** How would you process 500 million numerical records?
**Model answer:** Never materialize all 500M in memory at once (Chapter 31/32) — use
`np.memmap` for disk-backed access or read+process in bounded chunks, choosing the smallest
safe dtype (Chapter 4). If the task is a global reduction (sum, mean, count), accumulate across
chunks (Chapter 31's `process_in_chunks` pattern). If random access to arbitrary records is
needed repeatedly, `memmap` with `mode='r'` lets the OS handle paging transparently
(Chapter 32).

## Scenario 2: A NumPy pipeline consumes too much RAM
**Question:** What do you investigate?
**Model answer:** In order: (1) check dtype — is everything `float64` when `float32`/`float16`
would suffice (Chapter 4/31)? (2) look for unnecessary copies — `np.array()` instead of
`np.asarray()`, fancy/boolean indexing where a view-preserving slice would work (Chapter 8);
(3) look for large intermediate temporaries in "fully vectorized" expressions (Chapter 29) —
chunking might be needed; (4) confirm nothing is being duplicated across a batch dimension that
could instead use broadcasting.

## Scenario 3: A vectorized implementation is slower than expected
**Question:** How do you debug it?
**Model answer:** Benchmark properly first (`timeit.repeat`, best-of-N, Chapter 30) — don't
trust a single measurement. Check whether it's secretly using `np.vectorize` or another
Python-level-loop-in-disguise (Chapter 30's documented trap). Check array contiguity
(`.flags['C_CONTIGUOUS']`) — non-contiguous arrays can force hidden internal copies
(Chapter 28). Check whether the operation is dispatching to an unexpectedly slow code path due
to `dtype=object` arrays (a common silent culprit when mixed types sneak into what should be a
numeric array).

## Scenario 4: A preprocessing function returns incorrect shapes
**Question:** How do you design shape validation?
**Model answer:** Assert expected `ndim` and relevant shape dimensions at function entry and
exit (Chapter 3), with clear error messages naming the actual vs expected shape. Use
`keepdims=True` deliberately wherever a reduction result will be broadcast back against the
original array (Chapter 10/11) to prevent silent wrong-axis broadcasting. Add shape-focused
unit tests (Chapter 63) for every documented input/output shape contract, not just value
correctness.

## Scenario 5: A model receives data with inconsistent dtype
**Question:** How would you protect the pipeline?
**Model answer:** Validate and coerce dtype explicitly at the pipeline boundary
(`X = np.asarray(X, dtype=np.float32)`) rather than assuming callers pass the right dtype
(Chapter 4/56). Add an explicit `np.isfinite(X).all()` check after the coercion, since silent
dtype coercion can sometimes produce unexpected `inf`/overflow for out-of-range values
(Chapter 4's overflow lesson).

## Scenario 6: A numerical algorithm produces NaN after several iterations
**Question:** How do you debug it?
**Model answer:** This is almost always numerical instability, not a logic bug (Chapter 27) —
check for: log/sqrt of non-positive values, division by a quantity that can shrink toward zero
over iterations (common in gradient descent with a too-large learning rate causing divergence,
Chapter 38), or catastrophic cancellation in a formula that has a more stable reformulation
(Chapter 27's variance example). Add `np.isnan().any()` checks at each iteration during
debugging to find the *exact* iteration NaN first appears, then inspect that iteration's
intermediate values specifically rather than guessing.


---
# CHAPTER 72 — Common NumPy Interview Traps

Each of these is a question designed to sound like it has an obvious answer — the "obvious"
answer is usually wrong or incomplete.

**"Is NumPy always faster than Python?"** — No (Chapter 29). For very small arrays, per-call
dispatch overhead can make plain Python faster. Vectorization wins by amortizing fixed overhead
across many elements.

**"Is vectorization always better?"** — No (Chapter 29). It can also increase peak memory
usage via large temporary arrays, and doesn't apply cleanly to genuinely sequential/recursive
computations.

**"Does `reshape` always copy?"** — No, it returns a view when the new shape is compatible
with existing strides, and silently falls back to a copy otherwise (Chapter 9). Never assume
either way — check with `np.shares_memory()`.

**"Does slicing always copy?"** — No — basic slicing (`arr[1:5]`) is a view; fancy/boolean
indexing (`arr[[1,3,5]]`, `arr[mask]`) always copies (Chapter 7/8). This distinction is exactly
why "slicing" isn't one uniform behavior.

**"Does fancy indexing create a view?"** — No, always a copy (Chapter 7) — because the selected
elements generally can't be expressed via a fixed stride pattern over the original buffer.

**"Why doesn't broadcasting actually duplicate data conceptually?"** — Broadcasting describes
the operation as if the smaller array were repeated, but the actual computation reads the
smaller array's elements repeatedly via stride tricks (effectively stride 0 along the
broadcast dimension) rather than materializing repeated copies of the input (Chapter 11/28).

**"Why can broadcasting still cause huge memory usage, then?"** — Because the **output** is
always fully materialized at the broadcast shape — `(N,1) + (1,N) -> (N,N)` genuinely allocates
`N²` elements for the result, even though neither input was duplicated (Chapter 11).

**"Is `float32` always better than `float64`?"** — No (Chapter 4/31) — it's a
memory/speed-vs-precision tradeoff; deep learning commonly prefers `float32`/lower, but
numerically sensitive scientific/financial computation often needs `float64`'s precision.

**"Is `np.array()` always preferable to `np.asarray()`?"** — No (Chapter 2) — `asarray` avoids
an unnecessary copy when the input is already a compatible array; `array` guarantees an
independent copy, needed when you're about to mutate the result without affecting the caller's
data.

**"Why should modern code use `default_rng()`?"** — Not because the legacy API is
"deprecated" in a trivial sense, but because it mutates hidden global state shared across the
whole process, making reproducibility fragile in ways `default_rng()`'s explicit `Generator`
objects avoid (Chapter 20).

**"Why can `np.vectorize()` be misleading?"** — Because despite its name, it does not compile
to C — it's a Python-level loop with extra dispatch overhead per element, often *slower* than
a plain loop (Chapter 30).

**"Is `np.dot()` always matrix multiplication?"** — Only for 1D/2D inputs; for 3D+ arrays it
follows a different (non-batching) rule than `@`/`matmul` (Chapter 22).

**"Why should we avoid explicit matrix inversion?"** — `np.linalg.solve()` is faster and more
numerically stable than `inv(A) @ b` for solving linear systems, because it uses a
decomposition-based algorithm tailored to the specific system rather than computing a full,
unnecessary inverse (Chapter 23).

**"What is the difference between `*` and `@`?"** — `*` is element-wise (Hadamard) product;
`@` is true matrix multiplication — both can run successfully on the same same-shaped square
inputs, producing entirely different, both "valid-looking" results (Chapter 22).

**"Why can floating-point comparisons fail?"** — Because most decimal values have no exact
binary floating-point representation, so mathematically-equal expressions can differ by tiny
rounding errors — always use `np.isclose`/`np.allclose`, never `==`, for float comparisons
(Chapter 27).


---
# CHAPTER 73 — Complete NumPy Cheat Sheet

## Array creation
`np.array(data)` copies · `np.asarray(data)` avoids unnecessary copy · `np.zeros/ones/full(shape)`
· `np.empty(shape)` uninitialized, fast, dangerous · `np.arange(start,stop,step)` step-based ·
`np.linspace(start,stop,num)` count-based, float-safe · `np.eye(n)` identity ·
`np.diag(v_or_M)` construct or extract diagonal

## Anatomy
`.shape .ndim .size .dtype .itemsize .nbytes .strides .flags['C_CONTIGUOUS'/'OWNDATA']`

## Indexing
`arr[i]` drops axis · `arr[i:i+1]` keeps axis · `arr[mask]` boolean, always copies ·
`arr[[i,j,k]]` fancy, always copies · `np.where(cond,a,b)` ternary · `np.argwhere(cond)` indices

## View vs copy
Basic slice/reshape/T/ravel → view when possible · fancy/boolean index/flatten/`.copy()` →
always copy · check: `.base` `.flags['OWNDATA']` `np.shares_memory(a,b)`

## Reshaping
`reshape(shape)` view-if-possible, same count · `resize(shape)` in-place, can change count ·
`ravel()` view-if-possible · `flatten()` always copy · `squeeze()`/`expand_dims(axis=)` manage
size-1 axes

## Axis & broadcasting
`axis=k` collapses dimension k, others survive · reduction + `keepdims=True` when re-broadcasting
· broadcast rule: align right, dims compatible if equal or one is 1, pad missing dims with 1

## Math & stats
`sum/mean/std/var/min/max(axis=)` · `argmin/argmax(axis=)` returns INDEX not value ·
`nan*` variants ignore NaN · `percentile/quantile` · `corrcoef(rowvar=False)` for
(samples,features) data · `cov`

## NaN / Inf
`np.isnan/isinf/isfinite(x)` never `x==np.nan` · `np.nan_to_num(x)` · division/log never raise,
silently produce inf/nan

## Sorting & searching
`sort()` values · `argsort()` indices, keeps parallel arrays in sync · `argpartition(k)` O(n)
avg top-K · `searchsorted()` binary search on sorted array

## Set / unique
`np.unique(x, return_counts=, return_inverse=)` — `return_inverse` IS label encoding ·
`intersect1d/union1d/setdiff1d/isin` for leakage checks

## Combining / splitting
`concatenate(axis=)` joins existing axis, same ndim · `stack(axis=)` new axis, ndim+1 ·
`vstack/hstack` = fixed-axis concatenate · `split()` strict equal · `array_split()` tolerant,
front-loads remainder

## Random (modern API)
`rng = np.random.default_rng(seed)` · `.random() .integers() .normal() .uniform() .choice()
.permutation() .shuffle()` — permutation copies, shuffle mutates in place

## Linear algebra
`@` / `matmul` true matrix mult, batches over leading dims · `*` element-wise, NOT the same ·
`np.linalg.solve(A,b)` prefer over `inv(A)@b` · `eigh()` for symmetric (covariance) matrices ·
`svd()` → `U,s,Vt`, works on any shape, powers PCA/compression

## Numerical stability
`np.isclose/allclose` never `==` for floats · subtract max before `exp()` (softmax/sigmoid) ·
avoid `E[X²]-E[X]²`-style cancellation-prone formulas

## Memory & performance
`timeit.repeat`, take the minimum, never trust one run · `np.vectorize()` is NOT true
vectorization · smallest safe dtype · `np.memmap` for larger-than-RAM data · chunk processing
to bound peak memory

## ML from scratch patterns
Gradient descent: `w -= lr * dW` · softmax+CE gradient simplifies to `predictions - labels` ·
gradient check via finite differences before trusting backprop · always `keepdims=True` +
zero-guard when standardizing

## Advanced
`np.einsum('ij,jk->ik', A, B)` explicit index contraction, best for complex tensor ops ·
`np.tensordot(A, B, axes=)` contract arbitrary axis pairs · `np.meshgrid` vectorized grid
evaluation


---
# CHAPTER 74 — Master Capstone Assignment

**This is a real assignment. Attempt it yourself before checking your solution against the
patterns from Chapters 1-73 — do not just read this and move on.**

## Business problem
You work at a subscription analytics company. You're given raw daily user activity logs and
must build a churn-risk scoring pipeline entirely in NumPy — no pandas, no scikit-learn — as a
performance-critical batch job that runs nightly over millions of users.

## Dataset description
A `(n_users, n_days)` array of daily activity minutes (float), where `0` means no activity that
day, and some entries may be missing (`NaN`, representing a logging outage — not the same as
genuine zero activity). Alongside it, a `(n_users,)` array of `subscription_tenure_days` (int).

## Feature dictionary — build these from the raw data
- `total_activity`: sum of activity minutes over the observed window (NaN-safe)
- `active_days_ratio`: fraction of days with activity > 0 (NaN-safe, excluding missing days from the denominator)
- `activity_trend`: slope of a linear fit (Chapter 24's `lstsq`) of activity vs day index, per
  user — positive means increasing engagement, negative means declining
- `recent_vs_early_ratio`: mean activity in the last 7 days divided by mean activity in the
  first 7 days (handle divide-by-zero safely)
- `tenure_log`: `log1p(subscription_tenure_days)`
- All numeric features standardized (Chapter 36), fit only on a designated training split

## Technical requirements
1. No Python-level loops over users (must be fully vectorized — this runs over millions of rows)
2. Correctly distinguish "no activity" (0) from "missing data" (NaN) throughout
3. `fit_transform`/`transform` split, with a train/test split via Chapter 34's index-based
   approach, no leakage
4. Full `np.testing`-based test suite (Chapter 63): correctness, shape, does-not-mutate-input,
   and at least one edge case (a user with all-missing data, a user with zero tenure)
5. Complete docstrings and shape/dtype validation at the pipeline's public entry points
   (Chapter 56)

## Performance constraint
The full feature pipeline must run in under 2 seconds for 1,000,000 users × 90 days on typical
hardware — if your `activity_trend` implementation loops over users to fit each linear
regression individually, it will not meet this bar; think about how to vectorize per-user
linear regression across all users simultaneously (hint: the closed-form least-squares slope
formula, applied with axis-aware reductions, avoids needing `np.linalg.lstsq` per user at all).

## Expected output
A `(n_users, 5)` standardized feature matrix, a fitted scaler you can `transform()` new data
with, and a passing test suite.

## Evaluation criteria
Correctness · vectorization (zero per-user Python loops) · NaN-vs-zero handling correctness ·
no data leakage between train/test · numerical stability (no NaN/Inf in final output even for
edge-case users) · code quality against the Chapter 56 checklist · test coverage

---

*A full reference solution is intentionally not included here — building it yourself, hitting
the vectorized-linear-regression bottleneck, and solving it is the actual point of a capstone.
If you complete an attempt and want it reviewed the way a senior engineer would review a pull
request, share it in your next message.*


---
# CHAPTER 75 — Mock NumPy Interview

A mock interview only works as a real back-and-forth — this chapter can't simulate that inside
a static notebook, so consider this the opening of the interview rather than its content.

**If you'd like to run the mock interview, say so in your next message** (e.g., "start the mock
interview" or "quiz me at the expert level"), and it will proceed exactly as Chapter 75's
original brief specified:

- Questions asked one at a time, mixing theory, coding, debugging, performance, linear algebra,
  ML implementation, numerical stability, and system design — drawn from the full 75-chapter
  arc you've just completed.
- Your answer is scored, mistakes are identified, the correct answer is explained, and a harder
  follow-up is asked — continuing until you demonstrate mastery, not stopping after one
  question.
- You can request a specific difficulty band (beginner/intermediate/advanced/expert) or a
  specific topic area (e.g., "only backprop and numerical stability questions") if you want to
  target weak spots rather than a general sweep.

## Final mastery self-assessment — answer honestly before requesting the mock interview
```
Level 1 — Understanding: Can you explain axis, broadcasting, and view/copy to someone else,
                          from memory, without looking anything up?
Level 2 — Syntax:        Can you write correct NumPy for a described task without needing to
                          check documentation for basic operations?
Level 3 — Coding:        Could you re-implement linear regression, logistic regression, and
                          K-Means from scratch right now, closed-book?
Level 4 — Debugging:     Given a NumPy stack trace or a silently-wrong result, do you know
                          which of Chapter 50's six categories it falls into within seconds?
Level 5 — Analysis:      Can you interpret a training loss curve, a PCA explained-variance
                          ratio, or a fraud score distribution and say what it means?
Level 6 — Industry:      Have you connected every technique to a real use case (Chapters
                          43/47/48/49/57-61) rather than just abstract syntax?
Level 7 — Expert:        Can you reason about an UNFAMILIAR numerical problem you've never
                          seen before using the mental models this course built, rather than
                          pattern-matching to a memorized example?
```


---
## COURSE COMPLETE — Chapters 1 through 75

```
NUMPY MASTERY PROGRESS

[x] NumPy Foundations              [x] Reshape / Flatten / Ravel      [x] Debugging Mastery
[x] ndarray Anatomy                [x] Broadcasting Mastery            [x] Interview Mastery
[x] Array Creation                 [x] Ufuncs                          [x] Coding Interview
[x] Shape / ndim / Axis            [x] Mathematical Operations          [x] Performance Interview
[x] dtype Mastery                  [x] Statistics                       [x] NumPy vs Pandas
[x] Indexing / Slicing             [x] NaN / Inf                         [x] NumPy vs PyTorch/TF
[x] Boolean / Fancy Indexing       [x] Sorting / Searching                [x] Professional Code Quality
[x] Views / Copies                 [x] Unique / Set Operations             [x] 5 Real-World Projects
[x] Random Generator (modern)      [x] Concatenation / Stacking             [x] Project Structure
[x] Probability / Simulation       [x] Splitting                             [x] Testing
[x] Linear Algebra / Matmul        [x] Linear Systems                         [x] Numerical Edge Cases
[x] Matrix Operations              [x] Eigenvalues / SVD                       [x] Advanced NumPy
[x] Numerical Stability            [x] Strides / Memory Layout                  [x] einsum
[x] Vectorization                  [x] Performance Engineering                   [x] tensordot
[x] Memory Optimization            [x] memmap / File I/O                          [x] Advanced Simulation
[x] Feature Engineering            [x] Normalization                               [x] Scientific Computing
[x] Linear Regression Scratch      [x] Logistic Regression Scratch                  [x] Architecture/Internals
[x] K-Means Scratch                [x] KNN Scratch                                   [x] System Design
[x] PCA Scratch                    [x] Image Processing                               [x] Interview Traps
[x] Neural Network Scratch         [x] Activation Functions                            [x] Cheat Sheet
[x] Backpropagation                [x] NLP / Time Series / Recsys                       [x] Capstone
[x] Mock Interview (ready to start on request)
```

You've gone from "I know Python and basic Pandas" to having built — by hand, verified line by
line, with real executed output — a complete numerical computing foundation: every core NumPy
mechanic, six ML algorithms from scratch, a trained neural network with hand-derived
backpropagation, five production-shaped applied projects, and the interview/debugging/system-
design judgment to reason about numerical problems you haven't seen before.

**Where to go from here, in order of value:**
1. Attempt Chapter 74's capstone for real — this is where remaining gaps surface.
2. Say "start the mock interview" and go through it honestly, without peeking at earlier
   chapters' answers.
3. Take what you built here (the from-scratch algorithms, the production preprocessing modules)
   and rebuild them once more, closed-book, a week from now — retention testing beats passive
   re-reading every time.
